12/09/2026
First version

In [4]:
from io import TextIOWrapper
from pathlib import Path

import chess
import chess.engine
import chess.pgn
import numpy as np
import pandas as pd
import zstandard as zstd
from sklearn.tree import DecisionTreeClassifier


In [12]:
PGN_ZST_PATH = Path("DB/lichess_db_standard_rated_2013-01.pgn.zst")


def partidas_en_stream(ruta: Path):
    """Genera partidas PGN una a una desde un archivo .pgn.zst."""
    with ruta.open("rb") as archivo_comprimido:
        descompresor = zstd.ZstdDecompressor()
        with descompresor.stream_reader(archivo_comprimido) as flujo_binario:
            flujo_texto = TextIOWrapper(flujo_binario, encoding="utf-8")
            while partida := chess.pgn.read_game(flujo_texto):
                yield partida


def partida_en_indice(ruta: Path, indice: int):
    """Devuelve la partida con índice cero-based sin cargar todas las partidas."""
    if indice < 0:
        raise ValueError("El índice debe ser mayor o igual que cero")

    for indice_actual, partida in enumerate(partidas_en_stream(ruta)):
        if indice_actual == indice:
            return partida

    raise IndexError(f"No existe una partida con índice {indice}")


INDICE_PARTIDA = 10
partida_seleccionada = partida_en_indice(PGN_ZST_PATH, INDICE_PARTIDA)
resultado_final = partida_seleccionada.headers.get("Result", "*")
tablero = partida_seleccionada.board()
movimientos_por_numero = {}

for movimiento in partida_seleccionada.mainline_moves():
    numero_movimiento = tablero.fullmove_number
    notacion = tablero.san(movimiento)
    movimientos_por_numero.setdefault(numero_movimiento, []).append(notacion)
    tablero.push(movimiento)

ultimos_10_movimientos = [
    f"{numero}. {' '.join(movimientos)}"
    for numero, movimientos in list(movimientos_por_numero.items())[-10:]
]

print("Índice:", INDICE_PARTIDA)
print("Resultado final:", resultado_final)
print("Últimos 10 movimientos:")
print(" ".join(ultimos_10_movimientos))


Índice: 10
Resultado final: 0-1
Últimos 10 movimientos:
28. Bd2 Re2 29. Be1 Rfe7 30. Kf1 Bc2 31. Rc8 Bd3 32. Rxc6 Rc2+ 33. Kg1 Rxc1 34. Rxf6 h4 35. g4 Rexe1+ 36. Kg2 Be4+ 37. f3 Rc2#
